# Fase 3 — Pré-processamentoEste notebook prepara a matriz que vai para a clusterização (próximaentrega) e gera a **Figura 3** (variância explicada pelo PCA). São quatropassos: `log1p` nas variáveis de cauda longa, padronização, peso por blocoe PCA.A comparação entre os escalonadores (`RobustScaler`, escala 1–5 e`StandardScaler`) está no apêndice `03b`. Resumindo: o `StandardScaler` éo único que faz o peso `1/√n` dar 33% para cada bloco, e é ele que usamos.

In [ ]:
import sysfrom pathlib import Pathimport numpy as npimport pandas as pdfrom sklearn.compose import ColumnTransformerfrom sklearn.decomposition import PCAfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import FunctionTransformer, StandardScalerRAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(RAIZ / "src"))from config import BASE_FINAL_CSV, DICIONARIO_CSV, DATA_PROCESSEDfrom estilo import aplicar_estilo, salvarfrom figuras import plot_orcamento_blocos, plot_variancia_pca, plot_pc1_pc2aplicar_estilo()print("Python:", sys.executable)   # conferir se é o Python que tem as bibliotecasbase = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})dic = pd.read_csv(DICIONARIO_CSV)

## 1. Subconjunto e variáveisAs features vêm do dicionário. A capital fica fora (`flag_sem_iegm`).O `log1p` vai nas taxas criminais e nas duas variáveis em reais, que têmcauda longa (ver notebook 02). Percentuais e notas ordinais ficam comoestão.

In [ ]:
bloco_de = dic.set_index("coluna")["bloco"]ORDEM_BLOCOS = ["criminalidade", "socioeconomico", "gestao"]features = sorted(dic.loc[dic["papel"] == "feature", "coluna"],                  key=lambda c: (ORDEM_BLOCOS.index(bloco_de[c]), c))modelagem = base[~base["flag_sem_iegm"]].reset_index(drop=True)log_cols = ([c for c in features if bloco_de[c] == "criminalidade"]            + ["pib_percapita", "renda_domiciliar_mediana"])demais_cols = [c for c in features if c not in log_cols]print(f"{len(modelagem)} municípios | {len(features)} features | "      f"log1p em {len(log_cols)}")

## 2. Transformação e padronização com o `sklearn`O `ColumnTransformer` aplica `log1p` + `StandardScaler` nas colunas decauda longa e só `StandardScaler` nas outras. O ajuste é feito nos 644municípios, e não na base inteira.

In [ ]:
pre = ColumnTransformer([    ("log", Pipeline([("log1p", FunctionTransformer(np.log1p)),                      ("z", StandardScaler())]), log_cols),    ("z", StandardScaler(), demais_cols),])Z = pd.DataFrame(pre.fit_transform(modelagem[features]),                 columns=log_cols + demais_cols)[features]Z.describe().round(2).loc[["mean", "std"]]

## 3. Peso por blocoSem peso, o bloco com mais colunas pesa mais na distância só por ter maiscolunas. Multiplicamos cada coluna por `1/√n`, onde `n` é o número decolunas do bloco dela. Assim os três blocos contribuem igual.Essa ideia vem da Análise Fatorial Múltipla (Escofier e Pagès, 1994), quereescala cada grupo de variáveis para nenhum grupo dominar. A AFM dividepelo primeiro autovalor do grupo; aqui dividimos por `√n`, que dá o mesmoresultado quando cada coluna tem variância 1, e isso o `StandardScaler`garante.

In [ ]:
n_bloco = pd.Series([bloco_de[c] for c in features]).value_counts()peso = pd.Series({c: 1 / np.sqrt(n_bloco[bloco_de[c]]) for c in features})W = Z * pesopeso.groupby(bloco_de).first().round(3)

In [ ]:
def orcamento(M):    """% da variância total que cada bloco tem."""    var = M.var(ddof=0)    return (var.groupby(M.columns.map(bloco_de)).sum()               .reindex(ORDEM_BLOCOS) / var.sum() * 100).round(1)fig = plot_orcamento_blocos(orcamento(Z), orcamento(W))salvar(fig, "figura_orcamento_blocos")

Sem peso, criminalidade fica com uns 41% da distância e gestão com 32%. Como peso, cada bloco fica com 33,3%.

## 4. Figura 3 — PCA e variância explicada

In [ ]:
pca = PCA().fit(W)fig = plot_variancia_pca(pca.explained_variance_ratio_)salvar(fig, "figura3_variancia_pca")

São necessárias **12 das 22 componentes** para explicar 80% da variância.Ou seja, os dados não se resumem a poucas dimensões. Isso quer dizer duascoisas:1. Os três blocos não repetem informação (se repetissem, poucas componentes   bastariam). É o mesmo que a matriz de Spearman mostrou.2. Para a próxima entrega: dados espalhados em muitas dimensões são difíceis   para clusterização por densidade. É provável que o DBSCAN marque quase   tudo como ruído. Fica anotado aqui antes de rodar.

## 5. O que as primeiras componentes representam

In [ ]:
cargas = pd.DataFrame(pca.components_[:3].T, index=features,                      columns=["PC1", "PC2", "PC3"]).round(2)fortes = cargas[(cargas.abs() >= 0.25).any(axis=1)].copy()fortes["bloco"] = [bloco_de[c] for c in fortes.index]fortes.sort_values("PC1", key=abs, ascending=False)

In [ ]:
escores = pca.transform(W)fig = plot_pc1_pc2(escores, modelagem["taxa_urbanizacao"],                   "taxa de urbanização (%)", pca.explained_variance_ratio_)salvar(fig, "figura_pc1_pc2")

A PC1 é um eixo de condição socioeconômica: alfabetização, urbanização,coleta de lixo, renda e esgoto têm as maiores cargas, todas positivas. APC2 separa as notas de saúde e educação do IEGM das taxas de roubo.No gráfico, os municípios formam uma nuvem contínua, sem grupos separados.Os clusters da próxima entrega vão ser cortes nesse gradiente, e a silhuetatende a ser baixa. É bom saber isso agora para não interpretar mal oresultado depois.

## 6. Matriz exportada`matriz_modelagem.csv` tem os 644 municípios já transformados, padronizadose com peso. É o arquivo que a clusterização vai usar.

In [ ]:
saida = W.copy()saida.insert(0, "codigo_ibge", modelagem["codigo_ibge"])saida.insert(1, "municipio", modelagem["municipio"])saida.to_csv(DATA_PROCESSED / "matriz_modelagem.csv", index=False)print(f"-> matriz_modelagem.csv: {saida.shape[0]} x {saida.shape[1]}")

In [ ]:
k80 = int(np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.80)) + 1pd.DataFrame({    "passo": ["subconjunto", "log1p", "padronização", "peso por bloco", "PCA"],    "decisão": [        f"{len(modelagem)} municípios (sem a capital)",        f"{len(log_cols)} colunas: taxas criminais e valores em R$",        "StandardScaler, ajustado neste subconjunto (ver apêndice 03b)",        "1/√n: " + ", ".join(f"{b} {peso[[c for c in features if bloco_de[c] == b][0]]:.3f}"                              for b in ORDEM_BLOCOS),        f"{k80} componentes para 80% da variância",    ],})